# Exploracion - Transferencias de automotores (DNRPA)

Etapa 1 del proyecto `pronostico-autos`.

Objetivo de este notebook:
1. Descargar y cargar los datos (via `src/download_data.py` y `src/data.py`).
2. Chequear calidad: nulos, duplicados, rango de fechas, provincias.
3. Construir la serie mensual de **Santa Fe** y mirarla.
4. Separar los ultimos 12 meses como test y graficar train vs test.

> Antes de correr: `python src/download_data.py`

In [ ]:
# --- Setup: permitir importar los modulos de src/ ---
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# La raiz del repo es la carpeta padre si estamos dentro de notebooks/.
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(raiz))

from src.data import load_clean, get_series, split_train_test

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

## 1. Carga y limpieza

`load_clean()` lee el CSV crudo (`utf-8-sig`) y devuelve un DataFrame tidy:
`fecha`, `provincia`, `cantidad`.

In [ ]:
datos = load_clean()
print("Filas:", len(datos))
print("Provincias:", datos["provincia"].nunique())
print("Rango:", datos["fecha"].min().date(), "->", datos["fecha"].max().date())
datos.head()

## 2. Chequeos de calidad

In [ ]:
# Nulos por columna
print("Nulos:")
print(datos.isna().sum())

# Duplicados (misma provincia + misma fecha)
dups = datos.duplicated(subset=["provincia", "fecha"]).sum()
print(f"\nDuplicados provincia+fecha: {dups}")

# Nos aseguramos de que cada provincia tenga un mes por periodo (serie continua)
print("\nCantidad de registros por provincia (min/mediana/max):")
print(datos.groupby("provincia")["fecha"].count().describe().loc[["min", "50%", "max"]])

## 3. Serie mensual de Santa Fe

`get_series()` arma la serie con un indice de fechas **continuo**. Si faltara algún mes, tira un error con las fechas faltantes (no rellena con 0).

In [ ]:
serie = get_series(datos, "Santa Fe")
print(f"Meses: {len(serie)}  ({serie.index.min().date()} a {serie.index.max().date()})")
serie.tail()

## 4. Split train / test

Ultimos **12 meses** -> test (2025-10 a 2026-09). El resto -> train.

In [ ]:
train, test = split_train_test(serie, test_months=12)
print(f"train: {len(train)} meses ({train.index.min().date()} a {train.index.max().date()})")
print(f"test:  {len(test)} meses  ({test.index.min().date()} a {test.index.max().date()})")

In [ ]:
# Grafico: serie completa, con el test marcado en otro color.
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(train.index, train.values, label="train", color="#1f77b4", lw=1)
ax.plot(test.index, test.values, label="test (12 meses)", color="#d62728", lw=2)
ax.set_title("Transferencias mensuales de automotores - Santa Fe")
ax.set_xlabel("Fecha")
ax.set_ylabel("Cantidad")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Observaciones

- La serie tiene estacionalidad clara (picos/valles que se repiten cada anio),
  lo que anticipa que un modelo estacional va a funcionar razonablemente.
- Hay tendencia y cambios de nivel a lo largo de los anios (crisis, pandemia, etc.).
- En la Etapa 2 medimos un modelo base "igual que el mismo mes del anio anterior"
  con MAE y MAPE sobre el test.